# Primary MOF synthesis extraction

Extract primary synthesis records with settings from `configs/positive_extraction.json`. Run document matching first. The extraction prompts are stored in `prompts/positive_system.txt` and `prompts/positive_user.txt`.

## Inputs and setup

Install `python -m pip install -e ".[mining,notebook]"` from the repository root. Run document matching before this notebook, then run the cells in order. All paths below are repository-relative.

| Input | Main workflow location | Preparation |
| --- | --- | --- |
| Document manifest, CSV | `results/extraction/document_manifest.csv` | Generated by notebook 02; contains `DOI`, `Main File`, and `SI File`. |
| Main articles, PDF | `data/local/articles/` | Add real article PDFs named from their DOI, for example `10.1021_jacs.2c09756.pdf`. |
| Supporting information, PDF or DOCX | `data/local/supporting_information/` | Add the matching SI, for example `10.1021_jacs.2c09756_SI.pdf`. |
| Extraction prompts, TXT | `prompts/positive_system.txt`, `prompts/positive_user.txt` | Included and selected by the configuration. |

For the included illustrative pair, run notebook 02 with its default configuration, then change `CONFIG` below to `ROOT / "configs/example_positive_extraction.json"`. For a few real papers, use the [three-paper input template](../Demo/03_api_demo/README.md). Set `start_row` and concurrency in the selected configuration; `start_row` is a zero-based manifest row offset.

Validation makes no model requests. Enable `RUN_EXTRACTION` for extraction or `RUN_BACKFILL` to recover CSV rows from existing JSON. The main workflow saves the CSV and JSON store under `results/extraction/positive/`; the included example uses `results/examples/mining/positive/`. Retain both for negative mining.

Implementation: [positive extraction](../src/mofinder/extraction/positive.py), [response schema](../src/mofinder/extraction/schemas.py), and [JSON backfill](../src/mofinder/extraction/backfill.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src/mofinder").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from mofinder.display import display_path, display_paths
from mofinder.extraction.positive import load_config, validate_inputs, run_from_config
from mofinder.extraction.backfill import backfill_from_json

CONFIG = ROOT / "configs/positive_extraction.json"
config = load_config(CONFIG)
RUN_EXTRACTION = False
RUN_BACKFILL = False


## Inspect local inputs

Validation reports missing files and extracted text lengths without making API calls.


In [ ]:
if config["manifest_file"].exists():
    report = validate_inputs(config)
    print(display_paths(report))
else:
    print(f"Create the document manifest first: {display_path(config['manifest_file'])}")


## Extract primary syntheses

Enable `RUN_EXTRACTION` when ready. Enter your API key in the hidden prompt when requested, or set `OPENAI_API_KEY` in your environment first. The configured model and concurrency are used directly.


In [ ]:
if RUN_EXTRACTION:
    import os
    from getpass import getpass

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ").strip()
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("An API key is required for this run.")

    extraction_report = run_from_config(CONFIG)
    print(extraction_report)


## Recover CSV rows from saved JSON

Backfill appends saved payloads for DOIs absent from the CSV. It makes no API calls.


In [ ]:
if RUN_BACKFILL:
    backfill_report = backfill_from_json(
        config["manifest_file"], config["json_out_dir"], config["csv_out"],
        flush_every=config["flush_every"], article_dir=config["article_dir"],
        si_dir=config["si_dir"], project_root=config["project_root"],
    )
    print(display_paths(backfill_report))
